In [30]:
import pandas as pd

df_ss2030 = pd.read_csv("row_data/Arabic Sentiment Analysis Dataset - SS2030.csv")
df_training_1 = pd.read_csv("row_data/training_data (1).csv")
df_training_2 = pd.read_csv("row_data/training_data.csv")

print("Loaded")

Loaded


توحيد أسماء الأعمدة

In [31]:
# تحويل أسماء الأعمدة إلى lowercase
df_ss2030.columns = df_ss2030.columns.str.lower()
df_training_1.columns = df_training_1.columns.str.lower()
df_training_2.columns = df_training_2.columns.str.lower()

# إعادة التسمية
df_ss2030 = df_ss2030.rename(columns={'text': 'tweet', 'sentiment': 'label'})
df_training_1 = df_training_1.rename(columns={'text': 'tweet', 'sentiment': 'label'})
df_training_2 = df_training_2.rename(columns={'text': 'tweet', 'sentiment': 'label'})

print(df_ss2030.columns)
print(df_training_1.columns)
print(df_training_2.columns)

Index(['tweet', 'label'], dtype='object')
Index(['tweet', 'sarcasm', 'label', 'dialect'], dtype='object')
Index(['tweet', 'sarcasm', 'label', 'dialect'], dtype='object')


 اختيار الأعمدة المهمة

In [32]:
df1 = df_ss2030[['tweet', 'label']].copy()
df2 = df_training_1[['tweet', 'label']].copy()
df3 = df_training_2[['tweet', 'label']].copy()

print(df1.shape, df2.shape, df3.shape)

(4252, 2) (12548, 2) (12548, 2)


توحيد الـ labels

In [33]:
def fix_label(label):
    label = str(label).strip().upper()

    if label in ['NEG', 'NEGATIVE', '0']:
        return 0
    elif label in ['NEU', 'NEUTRAL', '1']:
        return 1
    elif label in ['POS', 'POSITIVE', '2']:
        return 2
    else:
        return None

df1['label'] = df1['label'].apply(fix_label)
df2['label'] = df2['label'].apply(fix_label)
df3['label'] = df3['label'].apply(fix_label)

print(df1['label'].value_counts())
print(df2['label'].value_counts())
print(df3['label'].value_counts())

label
1    2436
0    1816
Name: count, dtype: int64
label
1    5747
0    4621
2    2180
Name: count, dtype: int64
label
1    5747
0    4621
2    2180
Name: count, dtype: int64


الدمج 

In [34]:
df_final = pd.concat([df1, df2, df3], ignore_index=True)

print("Shape after merge:", df_final.shape)
print(df_final['label'].value_counts())

Shape after merge: (29348, 2)
label
1    13930
0    11058
2     4360
Name: count, dtype: int64


حذف التكرار 

In [35]:
df_final = df_final.drop_duplicates(subset='tweet')

print("After duplicates removal:", df_final.shape)

After duplicates removal: (16700, 2)


حذف القيم null

In [36]:
# حذف null
df_final = df_final.dropna(subset=['tweet', 'label'])

# حذف النصوص الفارغة
df_final = df_final[df_final['tweet'].str.strip() != ""]

print(df_final.isnull().sum())
print("After null removal:", df_final.shape)

tweet    0
label    0
dtype: int64
After null removal: (16700, 2)


Basic Cleaning

In [37]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = text.strip()
    return text

df_final['clean_text'] = df_final['tweet'].apply(clean_text)

df_final[['tweet', 'clean_text']].head()

,tweet,clean_text
0,حقوق المرأة 💚💚💚 https://t.co/Mzf90Ta5g1,حقوق المرأة
1,RT @___IHAVENOIDEA: حقوق المرأة في الإسلام. ht...,RT حقوق المرأة في الإسلام
2,RT @saud_talep: Retweeted لجنة التنمية بشبرا (...,RT Retweeted لجنة التنمية بشبرا \n \n ما زال ...
3,RT @MojKsa: حقوق المرأة التي تضمنها لها وزارة ...,RT حقوق المرأة التي تضمنها لها وزارة العدل
4,RT @abm112211: ولي امر الزوجة او ولي الزوجة او...,RT ولي امر الزوجة او ولي الزوجة او ولي المراة...


Arabic Normalization

In [38]:
def normalize_arabic(text):
    text = str(text)
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'(.)\1+', r'\1', text)
    return text

df_final['clean_text'] = df_final['clean_text'].apply(normalize_arabic)

df_final[['clean_text']].head()

,clean_text
0,حقوق المراه
1,RT حقوق المراه في الاسلام
2,RT Retweted لجنه التنميه بشبرا \n \n ما زال ال...
3,RT حقوق المراه التي تضمنها لها وزاره العدل
4,RT ولي امر الزوجه او ولي الزوجه او ولي المراه ...


حذف التكرار مرة ثانية (بعد التنظيف)

In [40]:
df_final = df_final.drop_duplicates(subset='clean_text')

print("After final duplicates:", df_final.shape)

After final duplicates: (16257, 3)


حذف النصوص الفارغة بعد التنظيف

In [41]:
df_final = df_final[df_final['clean_text'].str.strip() != ""]

print(df_final.isnull().sum())

tweet         0
label         0
clean_text    0
dtype: int64


cheeck

In [43]:
print(df_final.shape)

print("\nLabel distribution:")
print(df_final['label'].value_counts())

print("\nLabel ratio:")
print(df_final['label'].value_counts(normalize=True))

(16257, 3)

Label distribution:
label
1    7858
0    6295
2    2104
Name: count, dtype: int64

Label ratio:
label
1    0.483361
0    0.387218
2    0.129421
Name: proportion, dtype: float64


DATA QUALITY CHECK

In [46]:
# 1. Shape
print("\n Shape:")
print(df_final.shape)

# 2. Columns
print("\n Columns:")
print(df_final.columns)

# 3. Missing values
print("\n Missing values:")
print(df_final.isnull().sum())

# 4. Empty text
print("\n Empty texts:")
empty_tweet = (df_final['tweet'].str.strip() == "").sum()
empty_clean = (df_final['clean_text'].str.strip() == "").sum()
print("Empty tweet:", empty_tweet)
print("Empty clean_text:", empty_clean)

# 5. Duplicates
print("\n Duplicates:")
print("Tweet duplicates:", df_final.duplicated(subset='tweet').sum())
print("Clean_text duplicates:", df_final.duplicated(subset='clean_text').sum())

# 6. Label distribution
print("\n Label distribution:")
print(df_final['label'].value_counts())

# 7. Label ratio
print("\n Label ratio:")
print(df_final['label'].value_counts(normalize=True))

# 8. Text length
df_final['length'] = df_final['clean_text'].apply(lambda x: len(x.split()))

print("\n Text length stats:")
print(df_final['length'].describe())

# 9. Very short texts
short_texts = (df_final['length'] <= 2).sum()
print("\n Very short texts (<=2 words):", short_texts)

# 10. Sample preview
print("\n Sample data:")
print(df_final[['clean_text', 'label']].sample(5))


 Shape:
(16257, 4)

 Columns:
Index(['tweet', 'label', 'clean_text', 'length'], dtype='object')

 Missing values:
tweet         0
label         0
clean_text    0
length        0
dtype: int64

 Empty texts:
Empty tweet: 0
Empty clean_text: 0

 Duplicates:
Tweet duplicates: 0
Clean_text duplicates: 0

 Label distribution:
label
1    7858
0    6295
2    2104
Name: count, dtype: int64

 Label ratio:
label
1    0.483361
0    0.387218
2    0.129421
Name: proportion, dtype: float64

 Text length stats:
count    16257.000000
mean        15.822107
std          9.645767
min          1.000000
25%          9.000000
50%         14.000000
75%         20.000000
max         59.000000
Name: length, dtype: float64

 Very short texts (<=2 words): 264

 Sample data:
                                              clean_text  label
5369   كاميرات الاعلام الروسي ترصد الدمار الذي احدثته...      0
16226  الطريقه الوحيده لتعبرك امك فيا انو تحكيا عالوا...      1
15752  RT امازون ارسلت ايميلات لبعض مستخدميها تطلب

In [47]:
df_final = df_final[['clean_text', 'label']]

حفظ الداتا النظيفة

In [48]:
df_final[['clean_text', 'label']].to_csv("final_dataset.csv", index=False, encoding='utf-8-sig')